# Parkinson's Disease Detection Using Deep Learning — Interactive Walkthrough

Runs cleanly in **VS Code** (local Python 3.11 environment) or **Google Colab** (GPU runtime).

> **Disclaimer:** Experimental research/educational pipeline only. Not clinically validated.

In [ ]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DEFAULT_KAGGLE_DATASET, PROCESSED_DATA_DIR, RAW_DATA_DIR, ensure_project_dirs
ensure_project_dirs()
print("PROJECT_ROOT:", PROJECT_ROOT)

## Stage 1 — Download, Audit, Deduplicate, and Split
Downloads the Kaggle dataset via `kagglehub`, verifies image/DICOM integrity, removes MD5 duplicates, and builds `data/processed/{train,validate,test}/{healthy,parkinson}`.

In [ ]:
import shutil
import kagglehub
from src.data.audit import run_dataset_audit

DATASET_SLUG = DEFAULT_KAGGLE_DATASET  # e.g. 'irfansheriff/parkinsons-brain-mri-dataset'
raw_target = RAW_DATA_DIR / DATASET_SLUG.replace("/", "__")

if not raw_target.exists() or not any(raw_target.iterdir()):
    cache_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    shutil.copytree(cache_path, raw_target, dirs_exist_ok=True)

audit_summary = run_dataset_audit(raw_dir=raw_target, processed_dir=PROCESSED_DATA_DIR)
print(json.dumps(audit_summary, indent=2))

## Stage 2 — Exploratory Data Analysis (EDA)
Generates class balance, image dimension, sample grid, and pixel intensity shortcut charts in `reports/eda/`.

In [ ]:
from src.data.eda import run_eda

eda_summary = run_eda()
print(json.dumps(eda_summary, indent=2))

## Stage 3 — Train & Evaluate CustomCNN, VGG16, and ResNet50
Runs the complete training, unshuffled test evaluation, comparison plots, error analysis, and Grad-CAM overlays.

In [ ]:
!python scripts/03_train_and_evaluate.py